# 🧮 数学补丁：导数与梯度
### 配套 第 5 课《梯度下降》· 吴恩达《Machine Learning Specialization》

---

**为什么有这份文件？** 第 5 课的梯度下降，核心是**导数（斜率）**。你数学基础不牢没关系——这份文件**从"一条直线的陡度"开始，一步步讲到"导数""偏导数"**，每个概念都配一张图、一段能跑的代码。

**怎么用**：当成"工具抽屉"，学到哪、补到哪。看不懂的先跳到代码跑一跑，图会帮你建立感觉。

> 老规矩：**先理解"它在干嘛"，再管"符号怎么写"。**

## 第 0 步：环境自检

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
# ---- 选择你电脑上存在的中文字体：Windows 用微软雅黑，Mac 用苹方，Linux 用 Noto ----
_candidates = ["Microsoft YaHei", "SimHei", "PingFang SC",
               "Noto Sans CJK JP", "Noto Sans CJK SC", "Droid Sans Fallback"]
_available = {f.name for f in font_manager.fontManager.ttflist}
_chosen = [c for c in _candidates if c in _available] or ["DejaVu Sans"]
plt.rcParams["font.sans-serif"] = _chosen
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["savefig.bbox"] = "tight"
print("本次使用中文字体：", _chosen)
print("✅ 环境就绪，可以继续啦！")

---

## 1. 直线的"斜率"：它有多陡

**斜率（slope）= 陡不陡**。算法很简单：**竖着上升了多少 ÷ 横着走了多少**。

- 向右**上升**的线，斜率是**正**的；
- 向右**下降**的线，斜率是**负**的；
- 越陡，斜率的绝对值越大。

看图：

In [ ]:
xs = np.linspace(0, 10, 100)
plt.figure(figsize=(7.2, 4.6))
for slope, col in [(0.5, "tab:green"), (1.5, "tab:blue"), (3.0, "tab:red"), (-1.0, "tab:purple")]:
    plt.plot(xs, slope * xs, color=col, linewidth=2.2, label=f"斜率 = {slope}")
plt.xlabel("x"); plt.ylabel("y")
plt.title("直线的斜率：越大越陡；负的就是往下走")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---

## 2. 曲线的斜率：切线

直线每一处的陡度都一样。但**曲线不一样**——它每个点的陡度都在变。

那"曲线在某一点的斜率"是什么意思？做法是：**在那一点轻轻地贴一条直线**，这条刚好贴住的线叫**切线（tangent）**。**切线的斜率，就是曲线在这一点的斜率。**

看下面：同一条曲线 y = x²，在不同点上贴的切线，斜率都不一样——左边往下、中间水平、右边往上。

In [ ]:
def f(x):  return x**2
def df(x): return 2*x          # f 的导数（后面会讲）

xs = np.linspace(-3, 3, 200)
plt.figure(figsize=(7.6, 4.8))
plt.plot(xs, f(xs), color="tab:blue", linewidth=2.6, label="曲线 y = x²")
for x0, col in [(-2, "tab:red"), (0, "tab:green"), (2, "tab:purple")]:
    y0, slope = f(x0), df(x0)
    tx = np.linspace(x0-1, x0+1, 10)
    plt.plot(tx, y0 + slope*(tx-x0), "--", color=col, linewidth=1.8,
             label=f"x={x0} 处切线，斜率={slope}")
    plt.scatter([x0], [y0], color=col, s=70, zorder=3)
plt.xlabel("x"); plt.ylabel("y")
plt.title("切线：曲线在某一点的“斜率”")
plt.legend(fontsize=9); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**👀 看**：x = −2 的切线往下（斜率 −4），x = 0 的切线水平（斜率 0），x = 2 的切线往上（斜率 4）。

**这三句话是梯度下降的全部秘密：**
- 斜率 **> 0**（曲线在上升）→ 往**左**是下坡；
- 斜率 **< 0**（曲线在下降）→ 往**右**是下坡；
- 斜率 **= 0**（水平）→ 到**谷底**了，不用再走。

---

## 3. 导数：就是"曲线在一点的斜率"

**导数（derivative）没有新东西，它就是"曲线在某一点的斜率"**，换个正式名字而已。写法有两种：

$$f'(x) \quad \text{或} \quad \frac{df}{dx}$$

- `f'(x)` 读作 "f 撇 x"；
- `df/dx` 读作 "df 比 dx"，意思是"y 的变化 ÷ x 的变化"。

**怎么用眼睛读导数：** 看那个点的切线就够了——往上翘是正，往下走是负，水平是零。

（其实不用背公式：导数就是在某点附近量一下"x 挪一点点，y 挪了多少"，下面代码用数值方法验证。）

In [ ]:
def f(x):  return x**2

def numeric_deriv(f, x, h=1e-5):
    # 数值求导：在 x 左右各挪一点点，看 y 变化了多少
    return (f(x + h) - f(x - h)) / (2 * h)

for x in [-2, -1, 0, 1, 2]:
    print(f"x={x:>3}: 数值导数 ≈ {numeric_deriv(f, x):>6.3f}    公式 2x = {2*x:>3}")

---

## 4. 几个够用就行的导数

不用背一堆，线性回归和梯度下降里**最常出现的就这几个**：

| 函数 | 导数 | 说明 |
|---|---|---|
| `常数 c` | `0` | 平坦的，没有斜率 |
| `x` | `1` | 45° 斜线 |
| `x²` | `2x` | 最常用 |
| `2x` | `2` | 一条直线，斜率处处为 2 |
| `(x − 3)²` | `2(x − 3)` | 第 5 课的 J(w) 就是它 |

**怎么记 `x²` 的导数是 `2x` 的**：把指数 2 拿下来当系数，指数减 1 → `2x¹ = 2x`。（这个小规则叫"幂法则"，够用一辈子了。）

---

## 5. 偏导数 ∂：两个变量时怎么办

到了 `J(w, b)`，有 **w 和 b 两个旋钮**。"偏导数"就是解决"同时对两个变量求导"的办法，思路特别简单：

> **只拧其中一个旋钮，把另一个按住别动，看 J 怎么变。**

- `∂J/∂w`（对 w 的偏导数）：**假装 b 是固定的**，只看 J 随 w 怎么变；
- `∂J/∂b`（对 b 的偏导数）：**假装 w 是固定的**，只看 J 随 b 怎么变。

那个像"卷卷的 d"的符号 `∂`（读"偏"）就是在提醒你："**我只是对所有变量里的一个求导哦。**"

把它想成：你在山上，`∂J/∂w` 告诉你"**只朝东西方向**走是上坡还是下坡"，`∂J/∂b` 告诉你"**只朝南北方向**走是啥情况"。两个合起来，才知道该往真下的方向迈腿。

---

## 6. 为什么梯度下降离不开导数

回到第 5 课那句话：

$$w := w - \alpha \frac{d}{dw} J(w)$$

- `dJ/dw`（导数）告诉我们：**当前是上坡还是下坡、有多陡**；
- 前面的**减号**让我们**朝反方向（下坡）**走；
- `α` 决定这一步迈多大；
- 走到**导数 = 0**（平坦）的地方，就自动停下来了——那正是谷底。

**所以：导数 = 指南针，α = 步长。**

---

## 🎒 小抄

| 符号 | 读法 | 意思 |
|---|---|---|
| 斜率 | — | 一条线有多陡 |
| 切线 | — | 曲线在某点"贴住"的直线 |
| `f'(x)` 或 `df/dx` | 导数 | 曲线在某点的斜率 |
| `x²` 的导数 | — | `2x` |
| `∂J/∂w` | 偏导数 | 只对 w 求导（按住 b 不动） |
| 导数 > 0 / < 0 / = 0 | — | 上坡 / 下坡 / 到谷底 |

---

**补完这些，第 5 课的梯度下降就再没有陌生的符号了。** 哪里卡住，随时回来问。